In [9]:
import httpx

In [2]:
url = "https://api.gios.gov.pl/pjp-api/v1/rest/station/findAll"

In [3]:
response = httpx.get(url)
response.status_code

200

In [4]:
data = response.json()

In [5]:
type(data)

dict

In [6]:
data.keys()

dict_keys(['@context', 'Lista stacji pomiarowych', 'meta', 'links', 'totalPages'])

In [7]:
stations = data["Lista stacji pomiarowych"]

In [8]:
type(stations)

list

In [9]:
stations[0]

{'Identyfikator stacji': 11,
 'Kod stacji': 'DsCzerStraza',
 'Nazwa stacji': 'Czerniawa',
 'WGS84 φ N': '50.912475',
 'WGS84 λ E': '15.312190',
 'Identyfikator miasta': 142,
 'Nazwa miasta': 'Czerniawa',
 'Gmina': 'Świeradów-Zdrój',
 'Powiat': 'lubański',
 'Województwo': 'DOLNOŚLĄSKIE',
 'Ulica': 'ul. Strażacka 7'}

In [10]:
stations[0].keys()

dict_keys(['Identyfikator stacji', 'Kod stacji', 'Nazwa stacji', 'WGS84 φ N', 'WGS84 λ E', 'Identyfikator miasta', 'Nazwa miasta', 'Gmina', 'Powiat', 'Województwo', 'Ulica'])

In [11]:
len(stations)

20

In [12]:
for station in stations:
    print(station["Nazwa miasta"])

Czerniawa
Dzierżoniów
Kłodzko
Legnica
Oława
Oleśnica
Osieczów
Polkowice
Szczawno-Zdrój
Wałbrzych
Wrocław
Wrocław
Wrocław
Wrocław
Wrocław
Zgorzelec
Grudziądz
Inowrocław
Toruń
Bydgoszcz


In [13]:
pages = data["totalPages"]

In [14]:
type(pages)

int

In [15]:
pages

15

In [16]:
data["links"]

{'next': 'https://api.gios.gov.pl/pjp-api/v1/rest/station/findAll?page=1&size=20',
 'last': 'https://api.gios.gov.pl/pjp-api/v1/rest/station/findAll?page=14&size=20',
 'prev': 'https://api.gios.gov.pl/pjp-api/v1/rest/station/findAll?page=0&size=20',
 'self': 'https://api.gios.gov.pl/pjp-api/v1/rest/station/findAll?page=0&size=20',
 'first': 'https://api.gios.gov.pl/pjp-api/v1/rest/station/findAll?page=0&size=20'}

In [17]:
def get_all_stations():
    stations = []

    response = httpx.get(url, timeout=10.0)
    response.raise_for_status()
    data = response.json()

    stations.extend(data["Lista stacji pomiarowych"])

    total_pages = data["totalPages"]

    for page in range(1, total_pages):
        response = httpx.get(
            url,
            params={"page": page, "size": 20},
            timeout=10.0,
        )
        response.raise_for_status()

        page_data = response.json()
        stations.extend(page_data["Lista stacji pomiarowych"])

    return stations

In [18]:
stations = get_all_stations()

In [19]:
len(stations)

288

In [20]:
stations[0]

{'Identyfikator stacji': 11,
 'Kod stacji': 'DsCzerStraza',
 'Nazwa stacji': 'Czerniawa',
 'WGS84 φ N': '50.912475',
 'WGS84 λ E': '15.312190',
 'Identyfikator miasta': 142,
 'Nazwa miasta': 'Czerniawa',
 'Gmina': 'Świeradów-Zdrój',
 'Powiat': 'lubański',
 'Województwo': 'DOLNOŚLĄSKIE',
 'Ulica': 'ul. Strażacka 7'}

In [21]:
stations[-1]

{'Identyfikator stacji': 20647,
 'Kod stacji': 'WmBraniePCKMOB',
 'Nazwa stacji': 'Braniewo, ul. PCK',
 'WGS84 φ N': '54.383907',
 'WGS84 λ E': '19.815018',
 'Identyfikator miasta': 2803,
 'Nazwa miasta': 'Braniewo',
 'Gmina': 'Braniewo',
 'Powiat': 'braniewski',
 'Województwo': 'WARMIŃSKO-MAZURSKIE',
 'Ulica': 'PCK 4'}

In [22]:
def filter_silesian_stations(stations):
    return [station for station in stations if station["Województwo"] == "ŚLĄSKIE"]

In [23]:
silesian_stations = filter_silesian_stations(stations)

In [24]:
len(silesian_stations)

31

In [25]:
silesian_cities = {
    station["Nazwa miasta"]
    for station in silesian_stations
    if station["Nazwa miasta"] is not None
}

In [26]:
sorted(silesian_cities)

['Bielsko-Biała',
 'Cieszyn',
 'Czerwionka-Leszczyny',
 'Częstochowa',
 'Dąbrowa Górnicza',
 'Gliwice',
 'Goczałkowice-Zdrój',
 'Godów',
 'Jastrzębie-Zdrój',
 'Katowice',
 'Knurów',
 'Lubliniec',
 'Myszków',
 'Pszczyna',
 'Racibórz',
 'Rybnik',
 'Sosnowiec',
 'Tarnowskie Góry',
 'Tychy',
 'Ustroń',
 'Wodzisław Śląski',
 'Zabrze',
 'Zawiercie',
 'Złoty Potok',
 'Żory',
 'Żywiec']

In [27]:
len(silesian_cities)

26

## Dane archiwalne GIOŚ

In [2]:
import httpx

In [3]:
import sys
sys.path.append("../src")

In [4]:
from smogcast.ingest.gios import get_archival_data

In [5]:
from smogcast.ingest.gios import get_archival_data_for_period

In [6]:
from smogcast.ingest.gios import generate_date_ranges

In [20]:
from smogcast.ingest.gios import get_archival_data_by_sensor

In [8]:
station_id = 814

url = f"https://api.gios.gov.pl/pjp-api/v1/rest/station/sensors/{station_id}"

response = httpx.get(url, timeout=30.0)

print(response.status_code)

200


In [9]:
sensor_data = response.json()
sensor_data.keys()

dict_keys(['@context', 'Lista stanowisk pomiarowych dla podanej stacji', 'meta', 'links', 'totalPages'])

In [10]:
sensors = sensor_data["Lista stanowisk pomiarowych dla podanej stacji"]

type(sensors)

list

In [11]:
len(sensors)

20

In [12]:
sensors[0]

{'Identyfikator stanowiska': 5357,
 'Identyfikator stacji': 814,
 'Wskaźnik': 'arsen w PM10',
 'Wskaźnik - wzór': 'As(PM10)',
 'Wskaźnik - kod': 'As(PM10)',
 'Id wskaźnika': 60}

In [13]:
pm_sensors = [
    sensor
    for sensor in sensors
    if sensor.get("Wskaźnik – kod") in {"PM10", "PM2.5"}
]

pm_sensors

[]

In [14]:
[
    (
        sensor.get("Identyfikator stanowiska"),
        sensor.get("Wskaźnik"),
    )
    for sensor in sensors
]

[(5357, 'arsen w PM10'),
 (5358, 'benzo(a)antracen w PM10'),
 (5359, 'benzo(a)piren w PM10'),
 (5360, 'benzo(b)fluoranten w PM10'),
 (5361, 'benzo(j)fluoranten w PM10'),
 (5362, 'benzo(k)fluoranten w PM10'),
 (5363, 'kadm w PM10'),
 (5366, 'dibenzo(a,h)antracen w PM10'),
 (5367, 'indeno(1,2,3-cd)piren w PM10'),
 (5369, 'nikiel w PM10'),
 (5370, 'tlenek azotu'),
 (5371, 'dwutlenek azotu'),
 (5372, 'tlenki azotu'),
 (5373, 'ozon'),
 (5375, 'ołów w PM10'),
 (5376, 'pył zawieszony PM10'),
 (5377, 'pył zawieszony PM10'),
 (5378, 'pył zawieszony PM2.5'),
 (5379, 'pył zawieszony PM2.5'),
 (5382, 'dwutlenek siarki')]

In [15]:
pm_sensors = [
    sensor
    for sensor in sensors
    if sensor.get("Wskaźnik") in {
        "pył zawieszony PM10",
        "pył zawieszony PM2.5",
    }
]

pm_sensors

[{'Identyfikator stanowiska': 5376,
  'Identyfikator stacji': 814,
  'Wskaźnik': 'pył zawieszony PM10',
  'Wskaźnik - wzór': 'PM10',
  'Wskaźnik - kod': 'PM10',
  'Id wskaźnika': 3},
 {'Identyfikator stanowiska': 5377,
  'Identyfikator stacji': 814,
  'Wskaźnik': 'pył zawieszony PM10',
  'Wskaźnik - wzór': 'PM10',
  'Wskaźnik - kod': 'PM10',
  'Id wskaźnika': 3},
 {'Identyfikator stanowiska': 5378,
  'Identyfikator stacji': 814,
  'Wskaźnik': 'pył zawieszony PM2.5',
  'Wskaźnik - wzór': 'PM2.5',
  'Wskaźnik - kod': 'PM2.5',
  'Id wskaźnika': 69},
 {'Identyfikator stanowiska': 5379,
  'Identyfikator stacji': 814,
  'Wskaźnik': 'pył zawieszony PM2.5',
  'Wskaźnik - wzór': 'PM2.5',
  'Wskaźnik - kod': 'PM2.5',
  'Id wskaźnika': 69}]

In [16]:
sensor_ids = [
    sensor["Identyfikator stanowiska"]
    for sensor in pm_sensors
]

sensor_ids

[5376, 5377, 5378, 5379]

In [17]:
sensor_id = sensor_ids[0]
sensor_id

5376

In [18]:
SENSOR_ARCHIVAL_URL = (
    f"https://api.gios.gov.pl/pjp-api/"
    f"v1/rest/archivalData/getDataBySensor/{sensor_id}"
)

In [ ]:
sensor_archival_data = response.json()

In [ ]:
type(sensor_archival_data)

In [ ]:
sensor_archival_data.keys()

In [ ]:
measurements = sensor_archival_data["Lista archiwalnych wyników pomiarów"]

type(measurements)

In [ ]:
len(measurements)

In [ ]:
measurements[0]

In [ ]:
from smogcast.ingest.gios import parse_archival_measurements

In [ ]:
measurements = parse_archival_measurements(sensor_archival_data)

measurements

In [21]:
data = get_archival_data_by_sensor(
    5376,
    "2026-08-20 10:00",
    "2026-08-20 12:00",
)

data

[{'Nazwa stacji': 'Katowice, ul. Kossutha',
  'Kod stanowiska': 'SlKatoKossut-PM10-1g',
  'Data': '2026-08-20 10:00:00',
  'Wartość': 10.9},
 {'Nazwa stacji': 'Katowice, ul. Kossutha',
  'Kod stanowiska': 'SlKatoKossut-PM10-1g',
  'Data': '2026-08-20 11:00:00',
  'Wartość': 10.6},
 {'Nazwa stacji': 'Katowice, ul. Kossutha',
  'Kod stanowiska': 'SlKatoKossut-PM10-1g',
  'Data': '2026-08-20 12:00:00',
  'Wartość': 11.4}]